# vgra-yolo background training (Save Version)

One notebook per Kaggle account, **run by the owner of that account** (Kaggle allows one person per account).
It trains up to two runs at once, one per GPU.

- **Version 1:** check `REF`, `RUNS`, `SMOKE` and `BATCH` in the first cell, then Save Version (Save & Run All) with GPU T4 x2 and Internet on.
- **Version 2, 3, ...:** attach the output of the previous version as input (Add Input), change nothing else, Save Version again.
  Each run continues from where it stopped. A finished run is not trained again.
- **When all runs show FINISHED:** create a dataset from the notebook output and share it with the project owner.

Each version stops cleanly before the 12-hour limit (time budget `train.max_hours` in `configs/base.yaml`).
Inputs needed: the image dataset and `vgra-index` (both must be readable by this account).

The weights cell downloads every pretrained file once before the runs start; never remove it (parallel downloads of the same file corrupt it).

Run plan: notebook A = B0 seeds 1 and 2; notebook B = V8 seeds 0 and 1 (YOLOv8s), both on tag `detectors-v2`;
notebook C = V8R seeds 0 and 1 (YOLOv8s + ResCBAM, protocol amendment v1.1) on tag `detectors-v3`;
notebook D = RD seed 0 (RT-DETR-l) and FR seed 0 (Faster R-CNN R50-FPN v2), protocol amendment v1.2 section 11.3,
on tag `detectors-v4`. Notebook D starts with one smoke version (`SMOKE = True`: 1 epoch on 10% of train, under an hour;
it only decides the batch fallback `BATCH`), then the full runs (`SMOKE = False`) as above. Both must finish by 2026-12-14.
Guide for account owners: docs/COAUTHOR_RUNS.md.


In [ ]:
# Settings: keep these the same in every version of this notebook (notebook D: SMOKE changes once, after the smoke version)
REPO = "https://github.com/nithoukhan1/vgra-yolo.git"
REF = "detectors-v5"  # fixed tag (A, B: "detectors-v2"; C: "detectors-v3"; D: "detectors-v4", from its 3rd full version "detectors-v5")
RUNS = [  # (config, seed, GPU); notebook D = extension detectors. Notebook A: B0 seeds 1, 2; B: V8 0, 1; C: V8R 0, 1
    ("configs/RD.yaml", 0, 0),  # RT-DETR-l
    ("configs/FR.yaml", 0, 1),  # Faster R-CNN R50-FPN v2
]
SMOKE = True  # True = timing run (1 epoch on 10% of train) that only decides BATCH; False = the full runs
BATCH = {}  # protocol 11.3 fallback, only after the configured batch ran out of memory: {"RD": 4} and/or {"FR": 2}

In [ ]:
# Clone the fixed code, install, check
%cd /kaggle/working
!rm -rf vgra-yolo
!git clone --quiet {REPO} vgra-yolo
%cd /kaggle/working/vgra-yolo
!git checkout --quiet {REF}
!git log -1 --oneline
import subprocess
head = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
ref = subprocess.run(["git", "rev-parse", "--verify", "--quiet", f"{REF}^{{commit}}"], capture_output=True, text=True).stdout.strip()
if not head or head != ref:  # e.g. the tag is not on GitHub yet: stop before anything runs
    raise RuntimeError(f"STOP: the code is not at {REF}. Is {REF} pushed to GitHub? Nothing has run.")
print(f"code at {REF}: {head[:7]}")
!pip install -q -r requirements.txt
!pip install -q -e .
!python scripts/00_check_setup.py

In [ ]:
# Download the pretrained weights and the AMP-check model once, before the runs start.
# Two runs downloading the same file at the same moment write it twice and one of them
# reads a broken file (this crashed B0_s0 in its first version).
!python -c "from ultralytics import RTDETR, YOLO; [YOLO(w) for w in ('yolo11s.pt', 'yolov8s.pt', 'yolo26n.pt')]; RTDETR('rtdetr-l.pt'); print('Ultralytics weights ready')"
!python -c "from torchvision.models.detection import FasterRCNN_ResNet50_FPN_V2_Weights as W; W.COCO_V1.get_state_dict(progress=False); print('torchvision weights ready')"

In [ ]:
# Train: one process per GPU, each with its own log; wait for both
import os, subprocess, sys, time
from datetime import datetime

os.makedirs("/kaggle/working/logs", exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d-%H%M")
procs = []
for cfg, seed, gpu in RUNS:
    exp = os.path.splitext(os.path.basename(cfg))[0]
    name = f"{exp}_s{seed}"
    log_path = f"/kaggle/working/logs/{name}{'_smoke' if SMOKE else ''}_{stamp}.log"
    cmd = [sys.executable, "scripts/03_train_detector.py", "--config", cfg, "--seed", str(seed),
           "--device", str(gpu), "--smoke" if SMOKE else "--auto-resume"]
    if exp in BATCH:
        cmd += ["--batch", str(BATCH[exp])]
    procs.append((name, log_path, subprocess.Popen(cmd, stdout=open(log_path, "w"), stderr=subprocess.STDOUT)))
    print(f"started {name} on GPU {gpu} | {' '.join(cmd[2:])} | log {log_path}", flush=True)
    time.sleep(60)  # the runs share one data file; do not write it at the same moment
EXIT = {}  # 0 = normal end (finished or stopped for time); anything else = crashed
for name, log_path, p in procs:
    EXIT[name] = p.wait()
    print(f"{name} ended with exit code {EXIT[name]}", flush=True)

In [ ]:
# Summary of this version
import glob, json

FALLBACK = {"RD": 4, "FR": 2}  # protocol 11.3: the only allowed change of batch
for cfg, seed, gpu in RUNS:
    exp = os.path.splitext(os.path.basename(cfg))[0]
    name = f"{exp}_s{seed}"
    logs = sorted(glob.glob(f"/kaggle/working/logs/{name}{'_smoke' if SMOKE else ''}_*.log"))
    text = open(logs[-1], errors="replace").read() if logs else ""
    found = sorted(glob.glob(f"/kaggle/working/runs/{name}{'_smoke*' if SMOKE else ''}/metrics.json"))
    code = globals().get("EXIT", {}).get(name, 0)
    oom = "CUDA out of memory" in text or "OutOfMemoryError" in text
    if code and not oom:  # metrics.json may be the copy restored from the previous version: not this version's result
        print(f"{name}: CRASHED (exit code {code}). Its saved epochs are kept. Send the project owner these last lines "
              f"of {logs[-1] if logs else 'the log'}:")
        print("".join(text.splitlines(keepends=True)[-25:]))
    elif found and not (oom and code):
        d = json.load(open(found[-1]))
        v = d.get("valid") or {}
        if SMOKE:
            status = "SMOKE RUN DONE"
        else:
            status = "FINISHED" if d.get("finished") else "STOPPED FOR TIME (attach this output, run the next version)"
        print(f"{name}: {status} | batch {d.get('batch')} | epochs {d.get('epochs_total')} | "
              f"valid mAP50 {v.get('map50')} | mAP50-95 {v.get('map50_95')}")
        if SMOKE:
            start = text.find("=== timing (smoke run) ===")
            print(text[start:].split("\n\n")[0].strip() if start >= 0 else "(no timing lines in the log)")
            print(f"batch {d.get('batch')} fits: keep BATCH as it is for {exp}.\n")
    elif oom:
        if exp in FALLBACK and exp not in BATCH:
            print(f"{name}: OUT OF MEMORY with the configured batch. Protocol 11.3 allows one fallback, only when the "
                  f"first epoch runs out of memory: add \"{exp}\": {FALLBACK[exp]} to BATCH and run this version "
                  f"again (smoke version, or version 1 of the full runs). Otherwise send the log to the project owner.")
        else:
            print(f"{name}: OUT OF MEMORY at batch {BATCH.get(exp)}. No further fallback: send the log to the project owner.")
    else:
        print(f"{name}: no metrics.json. Last lines of {logs[-1] if logs else 'the log'}:")
        if logs:
            print("".join(text.splitlines(keepends=True)[-25:]))